In [1]:
# ============================================================================
# Cell 1: 导入库
# ============================================================================
from vnpy.alpha.lab import AlphaLab
from vnpy.trader.constant import Interval
from vnpy.alpha import Segment, AlphaDataset
from vnpy.alpha.dataset.template import logger
import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb
from pathlib import Path
from datetime import datetime
import gc

In [2]:
# ============================================================================
# Cell 2: 配置
# ============================================================================
vt_index_symbol = "000300.SSE"
BASE_PATH = Path('D:/Aquant project/MF')
LAB_PATH = BASE_PATH / 'MF_lab'

# 获取 MF_Lab
lab = AlphaLab(str(LAB_PATH))

# 时间配置
start = datetime(2018, 1, 1)
end = datetime(2026, 5, 8)

# 训练 / 验证 / 测试
DATASET_NAME = 'v100'
n_quantiles = 30

In [3]:
# ============================================================================
# Cell 3: 加载数据集
# ============================================================================
dataset: AlphaDataset = lab.load_dataset(DATASET_NAME)

In [4]:
# ============================================================================
# Cell 4: 提取 LambdaRank 数据
# ============================================================================
logger.info('提取 LambdaRank 训练数据...')
X_train, y_train, meta_train, group_train = dataset.extract_lambdarank_data(
    Segment.TRAIN, n_quantiles=n_quantiles
)

logger.info('提取验证数据...')
X_valid, y_valid, meta_valid, group_valid = dataset.extract_lambdarank_data(
    Segment.VALID, n_quantiles=n_quantiles
)

logger.info('提取测试数据...')
X_test, y_test, meta_test, group_test = dataset.extract_lambdarank_data(
    Segment.TEST, n_quantiles=n_quantiles
)

2026-05-24 00:18:38 提取 LambdaRank 训练数据...
2026-05-24 00:18:38 TRAIN, 标签值: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
2026-05-24 00:18:38 TRAIN, X.shape=(437100, 65)
2026-05-24 00:18:38 提取验证数据...
2026-05-24 00:18:38 VALID, 标签值: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
2026-05-24 00:18:38 VALID, X.shape=(72600, 65)
2026-05-24 00:18:38 提取测试数据...
2026-05-24 00:18:38 TEST, 标签值: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
2026-05-24 00:18:39 TEST, X.shape=(95100, 65)


In [5]:
# ============================================================================
# Cell 5: 训练 LambdaRank 模型
# ============================================================================
logger.info('开始训练 LambdaRank 模型...')

# X_* 为 pd.DataFrame，lgb 自动识别列名
train_data = lgb.Dataset(X_train, label=y_train, group=group_train)
valid_data = lgb.Dataset(X_valid, label=y_valid, group=group_valid, reference=train_data)

params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [1, 3, 5, 7, 10, 20, 30],
    'label_gain': [i**2 - 1 for i in range(n_quantiles)],
    'lambdarank_truncation_level': 10,

    'num_leaves': 1024,
    'max_depth': -1,
    'min_data_in_leaf': 300,

    'learning_rate': 0.001,
    'feature_fraction': 0.88,
    'bagging_fraction': 0.87,
    'bagging_freq': 5,

    'lambda_l1': 30,
    'lambda_l2': 0.0,

    'boosting_type': 'gbdt',
    'device': 'gpu',
    'verbose': -1,
    'seed': 42,
    'num_threads': -1,
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, valid_data],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(100),
        lgb.log_evaluation(period=1),
    ]
)

logger.info(f'训练完成！最佳迭代轮数: {model.best_iteration}')
if model.best_score:
    logger.info(f'VALID BEST NDCG: {model.best_score['valid']}')

2026-05-24 00:18:39 开始训练 LambdaRank 模型...
[1]	train's ndcg@1: 0.366446	train's ndcg@3: 0.375915	train's ndcg@5: 0.376637	train's ndcg@7: 0.375748	train's ndcg@10: 0.379248	train's ndcg@20: 0.387098	train's ndcg@30: 0.395413	valid's ndcg@1: 0.289143	valid's ndcg@3: 0.31754	valid's ndcg@5: 0.324573	valid's ndcg@7: 0.329886	valid's ndcg@10: 0.333407	valid's ndcg@20: 0.344524	valid's ndcg@30: 0.359158
Training until validation scores don't improve for 100 rounds
[2]	train's ndcg@1: 0.424833	train's ndcg@3: 0.421475	train's ndcg@5: 0.417686	train's ndcg@7: 0.412131	train's ndcg@10: 0.409218	train's ndcg@20: 0.407343	train's ndcg@30: 0.411806	valid's ndcg@1: 0.290717	valid's ndcg@3: 0.305544	valid's ndcg@5: 0.313227	valid's ndcg@7: 0.317887	valid's ndcg@10: 0.321135	valid's ndcg@20: 0.336232	valid's ndcg@30: 0.352494
[3]	train's ndcg@1: 0.464421	train's ndcg@3: 0.448151	train's ndcg@5: 0.438581	train's ndcg@7: 0.430973	train's ndcg@10: 0.422025	train's ndcg@20: 0.414657	train's ndcg@30: 0.41

In [6]:
# ============================================================================
# Cell 6: 生成回测信号
# ============================================================================
logger.info('---在测试集上预测---')

predictions = model.predict(X_test, num_iteration=model.best_iteration)
logger.info(f'预测完成，预测样本数:{len(predictions)}')

signal = meta_test.with_columns([
    pl.Series('signal', predictions)
])

logger.info(f'siganl.shape: {signal.shape}')
logger.info('siganl:')
logger.info(signal.head(5))
logger.info(signal.tail(5))

2026-05-24 00:18:45 ---在测试集上预测---
2026-05-24 00:18:45 预测完成，预测样本数:95100
2026-05-24 00:18:45 siganl.shape: (95100, 3)
2026-05-24 00:18:45 siganl:
2026-05-24 00:18:45 shape: (5, 3)
┌─────────────────────┬─────────────┬───────────┐
│ datetime            ┆ vt_symbol   ┆ signal    │
│ ---                 ┆ ---         ┆ ---       │
│ datetime[μs]        ┆ str         ┆ f64       │
╞═════════════════════╪═════════════╪═══════════╡
│ 2025-01-02 00:00:00 ┆ 000001.SZSE ┆ -0.001606 │
│ 2025-01-02 00:00:00 ┆ 000002.SZSE ┆ 0.000598  │
│ 2025-01-02 00:00:00 ┆ 000063.SZSE ┆ -0.00028  │
│ 2025-01-02 00:00:00 ┆ 000100.SZSE ┆ -0.001365 │
│ 2025-01-02 00:00:00 ┆ 000157.SZSE ┆ -0.002896 │
└─────────────────────┴─────────────┴───────────┘
2026-05-24 00:18:45 shape: (5, 3)
┌─────────────────────┬────────────┬───────────┐
│ datetime            ┆ vt_symbol  ┆ signal    │
│ ---                 ┆ ---        ┆ ---       │
│ datetime[μs]        ┆ str        ┆ f64       │
╞═════════════════════╪════════════╪══════

In [7]:
# ============================================================================
# Cell 7: 保存模型和信号
# ============================================================================
MODEL_NAME = 'v100'
SIGNAL_NAME = 'v100'

lab.save_model(MODEL_NAME, model)
lab.save_signal(SIGNAL_NAME, signal)
logger.info('模型和信号已保存')

2026-05-24 00:18:45 模型和信号已保存


In [8]:
# ============================================================================
# Cell 8: 特征重要性
# ============================================================================
logger.info('---特征重要性---')

importance_df = pd.DataFrame({
    'feature': model.feature_name(),
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

logger.info('Top 20 :')
print(importance_df.head(20))

2026-05-24 00:18:45 ---特征重要性---
2026-05-24 00:18:45 Top 20 :
                  feature   importance
60          ma_divergence  1686.296714
29                   rstr  1324.524717
45  range_adjusted_amihud  1287.528879
38           abn_turnover   803.444044
58              macd_hist   608.471057
59             boll_width   603.675692
31           streverse_2m   368.664075
43             liq_amihud   272.407877
42     abn_turnover_accel   239.912021
62            daily_range   184.641383
51                   rmax   155.882926
47          vol_stability   142.638559
40                    gtr   120.866975
39           turnover_vol   108.861404
34               rvol_21d   106.797775
30           streverse_1m    94.361979
63       effective_spread    84.731464
41  turnover_ret_interact    82.707129
56           shadow_ratio    81.002439
49                  rskew    80.646499


In [9]:
# ============================================================================
# Cell 9: 查看完整重要性
# ============================================================================
with pd.option_context('display.max_rows', None):
    print(importance_df)

                      feature   importance
60              ma_divergence  1686.296714
29                       rstr  1324.524717
45      range_adjusted_amihud  1287.528879
38               abn_turnover   803.444044
58                  macd_hist   608.471057
59                 boll_width   603.675692
31               streverse_2m   368.664075
43                 liq_amihud   272.407877
42         abn_turnover_accel   239.912021
62                daily_range   184.641383
51                       rmax   155.882926
47              vol_stability   142.638559
40                        gtr   120.866975
39               turnover_vol   108.861404
34                   rvol_21d   106.797775
30               streverse_1m    94.361979
63           effective_spread    84.731464
41      turnover_ret_interact    82.707129
56               shadow_ratio    81.002439
49                      rskew    80.646499
61             boll_bandwidth    75.711364
37                       cmra    65.042350
36         